In [5]:
!unzip archive.zip

Archive:  archive.zip
  inflating: customers.csv           
  inflating: discounts.csv           
  inflating: employees.csv           
  inflating: products.csv            
  inflating: stores.csv              
  inflating: transactions.csv        


In [15]:
!pip install -q openai sentence-transformers faiss-cpu duckdb

import pandas as pd, numpy as np, duckdb, re
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import faiss

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)
MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
#MODEL = "z-ai/glm-5.2"

In [4]:
def llm_chat(prompt, system=None, temperature=1, top_p=0.95, max_tokens=4096, verbose_reasoning=False):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}, "reasoning_budget": max_tokens},
        stream=True
    )

    full_content = ""
    for chunk in completion:
        if not chunk.choices:
            continue
        reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
        if reasoning and verbose_reasoning:
            print(reasoning, end="")
        if chunk.choices[0].delta.content is not None:
            full_content += chunk.choices[0].delta.content
    return full_content

In [6]:
BASE = "/content/"

customers    = pd.read_csv(BASE + "customers.csv", low_memory=False)
discounts    = pd.read_csv(BASE + "discounts.csv")
employees    = pd.read_csv(BASE + "employees.csv")
products     = pd.read_csv(BASE + "products.csv")
stores       = pd.read_csv(BASE + "stores.csv")
transactions = pd.read_csv(BASE + "transactions.csv", low_memory=False)

print(customers.shape, products.shape, transactions.shape, stores.shape, discounts.shape, employees.shape)

(1643306, 9) (17940, 12) (6416827, 19) (35, 8) (181, 6) (404, 4)


In [7]:
con = duckdb.connect(database=":memory:")
con.register("customers", customers)
con.register("discounts", discounts)
con.register("employees", employees)
con.register("products", products)
con.register("stores", stores)
con.register("transactions", transactions)

print(con.execute("SELECT COUNT(*) AS rows FROM transactions").fetchdf())

      rows
0  6416827


In [8]:
products_clean = products.copy()
for col in ["Description EN", "Color", "Sizes", "Sub Category"]:
    products_clean[col] = products_clean[col].fillna("")

products_clean["search_text"] = (
    products_clean["Category"].astype(str) + " " +
    products_clean["Sub Category"].astype(str) + " " +
    products_clean["Description EN"].astype(str) + " " +
    products_clean["Color"].astype(str)
)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(products_clean["search_text"].tolist(), batch_size=256, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def vector_search(query, k=8):
    q = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, k)
    res = products_clean.iloc[idxs[0]].copy()
    res["similarity"] = scores[0]
    return res[["Product ID","Category","Sub Category","Description EN","Color","Sizes","Production Cost","similarity"]]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/71 [00:00<?, ?it/s]

In [9]:
SCHEMA_DESC = """
Tables (DuckDB SQL, use double quotes for column names with spaces):
customers("Customer ID", Name, Email, Telephone, City, Country, Gender, "Date Of Birth", "Job Title")
products("Product ID", Category, "Sub Category", "Description EN", Color, Sizes, "Production Cost")
stores("Store ID", Country, City, "Store Name", "Number of Employees", "ZIP Code", Latitude, Longitude)
employees("Employee ID", "Store ID", Name, Position)
discounts(Start, End, Discont, Description, Category, "Sub Category")
transactions("Invoice ID", Line, "Customer ID", "Product ID", Size, Color, "Unit Price", Quantity, Date,
             Discount, "Line Total", "Store ID", "Employee ID", Currency, "Currency Symbol", SKU,
             "Transaction Type", "Payment Method", "Invoice Total")

Joins:
transactions."Product ID" = products."Product ID"
transactions."Store ID" = stores."Store ID"
transactions."Customer ID" = customers."Customer ID"
transactions."Employee ID" = employees."Employee ID"

Rules:
- Never SELECT customers.Name, Email, or Telephone unless a specific Customer ID is given
- Date column is a timestamp string 'YYYY-MM-DD HH:MM:SS' — use CAST(Date AS TIMESTAMP) or strftime for filtering
- Always add LIMIT 200 unless aggregating (SUM/COUNT/AVG/GROUP BY)
"""

def generate_sql(question):
    prompt = f"""You are a SQL generator for a retail analytics database (DuckDB syntax).

{SCHEMA_DESC}

Return ONLY the raw SQL query. No explanation, no markdown, no backticks.

Question: {question}
SQL:"""
    sql = llm_chat(prompt, temperature=0.1, max_tokens=600)
    return re.sub(r"```sql|```", "", sql).strip()

def run_sql_query(question):
    sql = generate_sql(question)
    try:
        df = con.execute(sql).fetchdf()
        return sql, df
    except Exception as e:
        return sql, pd.DataFrame({"error": [str(e)]})

In [10]:
# ---------- 1. Customer Service Agent ----------
def customer_service_agent(query, customer_id):
    sql = f"""
    SELECT t."Invoice ID", t."Product ID", p."Description EN", t.Size, t.Color,
           t.Quantity, t."Line Total", t.Date, t."Transaction Type", s."Store Name"
    FROM transactions t
    JOIN products p ON t."Product ID" = p."Product ID"
    JOIN stores s ON t."Store ID" = s."Store ID"
    WHERE t."Customer ID" = {customer_id}
    ORDER BY t.Date DESC LIMIT 20
    """
    order_history = con.execute(sql).fetchdf()
    context = order_history.to_string(index=False) if not order_history.empty else "No order history found."

    prompt = f"""You are a friendly retail customer service assistant.
Answer the customer's question using ONLY their own order data below. Never mention other customers.

Customer question: {query}

Customer's order history:
{context}

Answer:"""
    return llm_chat(prompt, temperature=0.5, max_tokens=800)


# ---------- 2. Search Agent (customer product discovery) ----------
def search_agent(query):
    results = vector_search(query, k=8)
    context = results.to_string(index=False)
    prompt = f"""You are a product search assistant for a fashion retailer.
A customer is searching with this query: "{query}"

Here are the most relevant matching products:
{context}

Present the top 3-5 most relevant options in a friendly, natural way, mentioning category, color, sizes, and price range (Production Cost is internal cost — DO NOT mention it to the customer, instead say "starting at approx X" using cost*2.5 as an estimated retail price).

Answer:"""
    return llm_chat(prompt, temperature=0.6, max_tokens=800)


# ---------- 3a. Recommendation Agent — CUSTOMER framing ----------
def recommendation_agent_customer(query, customer_id):
    hist_sql = f"""
    SELECT p.Category, p."Sub Category", p.Color
    FROM transactions t JOIN products p ON t."Product ID" = p."Product ID"
    WHERE t."Customer ID" = {customer_id} LIMIT 20
    """
    history = con.execute(hist_sql).fetchdf()
    fav_category = history["Category"].mode()[0] if not history.empty else "Feminine"

    results = vector_search(query + " " + fav_category, k=8)
    context = results.to_string(index=False)

    prompt = f"""You are a personal shopping assistant recommending products to a customer.
Customer's request: {query}
Customer's most-purchased category: {fav_category}

Candidate products:
{context}

Recommend 3-4 items that best match their taste and request, explain briefly why each fits.

Answer:"""
    return llm_chat(prompt, temperature=0.6, max_tokens=800)

In [11]:
# ---------- 4. Inventory Agent ----------
def inventory_agent(query):
    sql, df = run_sql_query(query)
    context = df.to_string(index=False) if not df.empty else "No data."

    prompt = f"""You are an inventory optimization analyst for a fashion retailer.

Staff question: {query}

SQL used: {sql}

Data retrieved:
{context}

Provide inventory insights: identify stock risk, fast/slow movers, and a clear recommended action (restock, transfer, hold, discount).

Answer:"""
    return llm_chat(prompt, temperature=0.4, max_tokens=1000)


# ---------- 5b. Recommendation Agent — RETAILER framing ----------
def recommendation_agent_staff(query):
    sql = """
    SELECT p."Product ID", p.Category, p."Sub Category", p."Description EN",
           SUM(t.Quantity) AS units_sold, SUM(t."Line Total") AS revenue
    FROM transactions t JOIN products p ON t."Product ID" = p."Product ID"
    GROUP BY p."Product ID", p.Category, p."Sub Category", p."Description EN"
    ORDER BY units_sold ASC LIMIT 15
    """
    slow_movers = con.execute(sql).fetchdf()
    context = slow_movers.to_string(index=False)

    prompt = f"""You are a merchandising strategy assistant for retail staff.

Staff question: {query}

Slow-moving products (candidates for promotion/bundling):
{context}

Recommend which products to promote, bundle, or discount to boost sell-through, with reasoning.

Answer:"""
    return llm_chat(prompt, temperature=0.5, max_tokens=1000)


# ---------- 6. Executive Insights Agent ----------
def executive_insights_agent(query):
    kpi_sql = """
    SELECT s.Country, SUM(t."Line Total") AS revenue, COUNT(DISTINCT t."Invoice ID") AS orders,
           SUM(t.Quantity) AS units
    FROM transactions t JOIN stores s ON t."Store ID" = s."Store ID"
    GROUP BY s.Country ORDER BY revenue DESC LIMIT 10
    """
    kpis = con.execute(kpi_sql).fetchdf()
    sql, extra_df = run_sql_query(query)
    context = f"Top-line KPIs by country:\n{kpis.to_string(index=False)}\n\nQuery-specific data (SQL: {sql}):\n{extra_df.to_string(index=False)}"

    prompt = f"""You are an executive insights analyst presenting to retail leadership.

Leadership question: {query}

Data:
{context}

Summarize in a business-friendly way: key trend, risk, opportunity, and one recommended strategic action.

Answer:"""
    return llm_chat(prompt, temperature=0.4, max_tokens=1000)

In [12]:
ROLE_AGENT_MAP = {
    "customer": ["customer_service_agent", "recommendation_agent", "search_agent"],
    "retailer": ["inventory_agent", "recommendation_agent", "executive_insights_agent"]
}

def classify_intent(query, role):
    allowed = ROLE_AGENT_MAP[role]
    prompt = f"""Classify this query into exactly one of these allowed agents: {allowed}

Query: {query}

Agent definitions:
- customer_service_agent: order status, returns, past purchases, policy questions
- search_agent: finding/discovering products by description or need
- recommendation_agent: personalized product suggestions (customer) OR merchandising/promotion suggestions (retailer)
- inventory_agent: stock levels, restocking, stockout risk, replenishment
- executive_insights_agent: high-level business performance, revenue, trends, strategic summaries

Reply with only the exact agent name from the allowed list."""
    result = llm_chat(prompt, temperature=0, max_tokens=20)
    result = result.strip().lower()
    for agent in allowed:
        if agent in result:
            return agent
    return allowed[0]

def route_query(query, role, customer_id=None):
    if role not in ROLE_AGENT_MAP:
        return "Invalid role. Must be 'customer' or 'retailer'."

    agent = classify_intent(query, role)
    print(f"[Router] Role={role} -> Agent={agent}")

    if agent == "customer_service_agent":
        return customer_service_agent(query, customer_id)
    elif agent == "search_agent":
        return search_agent(query)
    elif agent == "recommendation_agent":
        return recommendation_agent_customer(query, customer_id) if role == "customer" else recommendation_agent_staff(query)
    elif agent == "inventory_agent":
        return inventory_agent(query)
    elif agent == "executive_insights_agent":
        return executive_insights_agent(query)

In [18]:
# Customer example
print(route_query("How is revenue trending across countries?", role="retailer"))

[Router] Role=retailer -> Agent=inventory_agent
**Unable to provide inventory insights — the query failed and the required data is missing.**

### Why this doesn't work:
1. **SQL Error**: `Binder Error: Referenced column "Country" not found in FROM clause`  
   → The column should be qualified as `s."Country"` (from the `stores` table), not just `"Country"`.

2. **Wrong data for inventory analysis**: Even if fixed, this query only returns **revenue by country/month** (`Line Total`, `Country`, `year_month`).  
   Inventory insights require:
   - Current stock levels (on-hand units)
   - Sell-through velocity (units sold per week)
   - Weeks of supply / cover
   - Aging / time in stock
   - Product/category attributes (seasonality, lifecycle)

---

### What you need to run instead:
A query that joins **transactions → stores → inventory → products**, e.g.:

```sql
WITH sales AS (
  SELECT
    t."Product ID",
    s."Country",
    strftime('%Y-%m', CAST(t."Date" AS TIMESTAMP)) AS year_month

In [ ]:
# Customer example - order history
print(route_query("What did I buy last time?", role="customer", customer_id=10142))

[Router] Role=customer -> Agent=customer_service_agent
Your most recent purchase was on **December 27, 2024** at our **Store New York** location:

- **Lace Blouse With 3/4 Sleeve** (Size S) – **$13.25**

(Invoice: INV-US-001-03836105)

Just before that, on December 23, you bought a pair of **Women's Cargo Pants (Size 38)** for $53.75, but that item was returned the same day.


In [ ]:
# Retailer example - inventory
print(route_query("Which products have the lowest units sold and might be overstocked?", role="retailer"))

[Router] Role=retailer -> Agent=inventory_agent
# Inventory Optimization Analysis: Fashion Retailer

## Executive Summary
Analysis of **3,847 products** reveals a **long-tail demand distribution** with significant polarization: 22% of SKUs sell ≤20 units (overstock risk), while 18% sell ≥200 units (stockout risk). Immediate action needed on 847 slow-moving and 692 fast-moving SKUs.

---

## 📊 Demand Velocity Segmentation

| Segment | Units Sold Range | SKU Count | % of Catalog | Revenue Risk | Action Priority |
|---------|------------------|-----------|--------------|--------------|-----------------|
| **Critical Overstock** | 6-15 units | 187 | 4.9% | High carrying cost, markdown liability | **URGENT: Discount/Liquidate** |
| **Slow Movers** | 16-30 units | 342 | 8.9% | Capital tied up, seasonal obsolescence | **HIGH: Promote/Transfer** |
| **Moderate Velocity** | 31-100 units | 1,234 | 32.1% | Healthy turn, monitor trends | **MEDIUM: Hold/Replenish** |
| **Fast Movers** | 101-250 uni

### Agent Evaluation Framework
This framework uses a set of test cases to measure:
- **Success Rate**: Did the code execute without error?
- **Faithfulness**: Did the agent hallucinate information not present in the data?
- **Relevance**: Did the agent actually address the user's specific query?

In [21]:
eval_test_cases = [
    {"role": "customer", "query": "What is the status of my last order?", "customer_id": 10142, "expected_intent": "customer_service_agent"},
    {"role": "customer", "query": "I need some blue summer dresses for a wedding", "customer_id": 47162, "expected_intent": "search_agent"},
    {"role": "retailer", "query": "Which stores in the US have the highest revenue this month?", "expected_intent": "executive_insights_agent"},
    {"role": "retailer", "query": "Show me products with zero sales in the last 30 days", "expected_intent": "inventory_agent"}
]

def evaluate_response(query, response, context=""):
    """Uses the LLM as a judge to score the response from 1-5."""
    eval_prompt = f"""Evaluate the following retail assistant response based on the User Query.

    User Query: {query}
    Assistant Response: {response}

    Score the response on a scale of 1 to 5 based on:
    1: Incorrect, hallucinated, or irrelevant.
    3: Correct but missing detail or slightly off-tone.
    5: Perfect, helpful, and grounded in data.

    Return ONLY a JSON object: {{"score": <int>, "reasoning": "<string>"}}"""

    eval_result = llm_chat(eval_prompt, temperature=0)
    try:
        import json
        return json.loads(re.search(r'\{.*\}', eval_result, re.DOTALL).group())
    except:
        return {"score": 0, "reasoning": "Failed to parse evaluation"}

results = []
for test in eval_test_cases:
    print(f"Testing: {test['query']}")
    actual_intent = classify_intent(test['query'], test['role'])

    try:
        response = route_query(test['query'], test['role'], test.get('customer_id'))
        eval_score = evaluate_response(test['query'], response)

        results.append({
            "query": test['query'],
            "expected_intent": test['expected_intent'],
            "actual_intent": actual_intent,
            "score": eval_score['score'],
            "reasoning": eval_score['reasoning']
        })
    except Exception as e:
        results.append({"query": test['query'], "error": str(e), "score": 0})

eval_df = pd.DataFrame(results)
display(eval_df)

print(f"Average Agent Score: {eval_df['score'].mean():.2f} / 5.0")
print(f"Intent Classification Accuracy: {(eval_df['expected_intent'] == eval_df['actual_intent']).mean() * 100:.1f}%")

Testing: What is the status of my last order?
[Router] Role=customer -> Agent=customer_service_agent
Testing: I need some blue summer dresses for a wedding
[Router] Role=customer -> Agent=customer_service_agent
Testing: Which stores in the US have the highest revenue this month?
[Router] Role=retailer -> Agent=inventory_agent
Testing: Show me products with zero sales in the last 30 days
[Router] Role=retailer -> Agent=inventory_agent


,query,expected_intent,actual_intent,score,reasoning
0,What is the status of my last order?,customer_service_agent,customer_service_agent,5,The response directly answers the user's query...
1,I need some blue summer dresses for a wedding,search_agent,customer_service_agent,3,The response correctly references the user's o...
2,Which stores in the US have the highest revenu...,executive_insights_agent,inventory_agent,1,The response fails to answer the user's questi...
3,Show me products with zero sales in the last 3...,inventory_agent,inventory_agent,1,The response is a hallucination. The user aske...


Average Agent Score: 2.50 / 5.0
Intent Classification Accuracy: 50.0%
